## Сбор физических параметров вратарей из Transfermarkt

In [1]:
import warnings
import unicodedata
import numpy as np
import pandas as pd
from thefuzz import process as fuzz_process

warnings.filterwarnings('ignore')

In [2]:
df_tm = pd.read_csv('../datasets/players.csv')
df_tm_gk = (df_tm[df_tm['position'] == 'Goalkeeper'][['name', 'height_in_cm']].dropna(subset=['name']).reset_index(drop=True))
print(f'Вратарей на Transfermarkt: {len(df_tm_gk)}')
print(f'Из них с ростом: {df_tm_gk["height_in_cm"].notna().sum()}')

Вратарей на Transfermarkt: 5521
Из них с ростом: 5054


In [3]:
def normalize(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode('ascii').strip().lower()


df_games = pd.read_csv('../datasets/goalkeepers_in_games.csv')
unique_gks = sorted(df_games['player'].dropna().unique())
print(f'Уникальных вратарей в играх: {len(unique_gks)}')

tm_exact = dict(zip(df_tm_gk['name'], df_tm_gk.index))
tm_norm  = {normalize(n): i for n, i in tm_exact.items()}

results = []
for player in unique_gks:
    height = np.nan

    if player in tm_exact:
        height = df_tm_gk.loc[tm_exact[player], 'height_in_cm']
    else:
        p_norm = normalize(player)
        if p_norm in tm_norm:
            height = df_tm_gk.loc[tm_norm[p_norm], 'height_in_cm']
        else:
            match, score = fuzz_process.extractOne(p_norm, list(tm_norm.keys()))
            if score >= 85:
                height = df_tm_gk.loc[tm_norm[match], 'height_in_cm']

    results.append({'keeper': player, 'height_cm': height})

df_physical = pd.DataFrame(results)

found = df_physical['height_cm'].notna().sum()
print(f'Найдено: {found}')

Уникальных вратарей в играх: 577
Найдено: 568


In [4]:
df_physical[df_physical['height_cm'].isna()]

,keeper,height_cm
6,Adrián,NaN
43,Andrei Radu,NaN
49,Andrés Prieto,NaN
164,Fabio Coltorti,NaN
242,Jeremias Ledesma,NaN
277,Jérémy Vachoux,NaN
455,René Adler,NaN
544,Vincent Demarconnay,NaN
553,Wilfredo Caballero,NaN


In [5]:
df_physical.to_csv('../datasets/gk_physical.csv', index=False)
df_physical

,keeper,height_cm
0,Aaron Ramsdale,190.0
1,Aarón Escandell,186.0
2,Abdoulaye Diallo,191.0
3,Adam Federici,188.0
4,Adrian Semper,194.0
...,...,...
572,Álvaro Fernández,185.0
573,Álvaro Valles,191.0
574,Ángel Fortuño,183.0
575,Édouard Mendy,194.0
